In [3]:
import os
import wandb
import torch

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize
from pytorch_lightning.plugins.environments import LightningEnvironment

from helpers import set_all_seeds, build_model, build_datamodule

# Patch TabularPlugin to fix DDP spawn crash where tabular_df is None on subprocesses
try:
    from udm.plugins.tabular.tabular_plugin import TabularPlugin
    if not hasattr(TabularPlugin, "_original_setup"):
        TabularPlugin._original_setup = TabularPlugin._setup        
        def patched_setup(self):
            if self.tabular_df is None:
                self.tabular_df = self.load_tabular_df()
            self._original_setup()
        TabularPlugin._setup = patched_setup
except ImportError:
    pass

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.finish()
    logger = WandbLogger(project=cfg.wandb.project, dir="wandb/")

    model = build_model(cfg)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=logger,
        accelerator="gpu",
        strategy="ddp_notebook" if cfg.devices > 1 else "auto",
        
        devices=cfg.devices,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision,
        enable_checkpointing=False,
        log_every_n_steps=1,
        plugins=[LightningEnvironment()],
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        sync_batchnorm=True if cfg.devices > 1 else False
    )

    trainer.fit(model, datamodule=datamodule)
    # trainer.test(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config" # _synthetic
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Global seed set to 420


/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
  rank_zero_warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/l ...
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | T

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.
